In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- Setup Chrome Driver ---
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_experimental_option("detach", True)  # Keep browser open
prefs = {"credentials_enable_service": False, "profile.password_manager_enabled": False}
chrome_options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=chrome_options
)

wait = WebDriverWait(driver, 20)  # wait for page loads

try:
    # --- 1️⃣ Go to Login Page ---
    driver.get("http://localhost:3000/auth")

    # --- 2️⃣ Enter Email & Password ---
    email_input = wait.until(
        EC.presence_of_element_located((By.XPATH, "//input[@type='email']"))
    )
    password_input = driver.find_element(By.XPATH, "//input[@type='password']")
    email_input.send_keys("sakib2333@gmail.com")
    password_input.send_keys("Password123")

    # --- 3️⃣ Click “Sign In” button ---
    sign_in_button = driver.find_element(
        By.XPATH, "//button[contains(text(), 'Sign In')]"
    )
    sign_in_button.click()

    # --- 4️⃣ Wait until dashboard page loads ---
    wait.until(EC.url_contains("/dashboard"))
    print("✅ Logged in successfully!")

    # --- 5️⃣ Go to Invoice Table Page first ---
    driver.get("http://localhost:3000/invoice")
    time.sleep(2)

    # --- 6️⃣ Go to Create Invoice Page ---
    driver.get("http://localhost:3000/invoice_create")

    # --- 7️⃣ Wait for Customer dropdown ---
    wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//select[contains(@class,'w-full')]")
        )
    )

    # --- 8️⃣ Select first existing customer ---
    customer_dropdown = Select(
        driver.find_element(By.XPATH, "//select[contains(@class,'w-full')]")
    )
    customer_dropdown.select_by_index(1)

    # --- 9️⃣ Select Product ---
    product_dropdown = Select(
        driver.find_element(
            By.XPATH, "//select[starts-with(@class,'border px-3 py-2')]"
        )
    )
    product_dropdown.select_by_index(1)

    # --- 🔟 Enter Quantity ---
    qty_input = driver.find_element(
        By.XPATH, "//input[@type='number' and not(@readonly)]"
    )
    qty_input.clear()
    qty_input.send_keys("2")

    # --- 1️⃣1️⃣ Click Add Product ---
    add_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Add')]")
    add_button.click()
    time.sleep(1)

    # --- 1️⃣2️⃣ Select Payment Method: Offline ---
    offline_button = driver.find_element(
        By.XPATH, "//button[contains(text(), 'Offline')]"
    )
    offline_button.click()

    # --- 1️⃣3️⃣ Fill Notes ---
    notes_input = driver.find_element(By.XPATH, "//textarea[@rows='3']")
    notes_input.send_keys("Test automation invoice note")

    # --- 1️⃣4️⃣ Click Proceed & Generate PDF ---
    proceed_button = driver.find_element(
        By.XPATH, "//button[contains(text(), 'Proceed & Generate PDF')]"
    )
    proceed_button.click()

    # --- 1️⃣5️⃣ Wait until invoice page loads (new URL) ---
    wait.until(
        lambda d: "/invoice/" in d.current_url
        and "/invoice_create" not in d.current_url
    )
    invoice_url = driver.current_url
    print(f"✅ Invoice generated! Current URL: {invoice_url}")

    # Extract invoice ID from URL
    invoice_id = invoice_url.split("/")[-1]

    # --- 1️⃣6️⃣ Click Floating Share Button twice (Next.js version) ---
    try:
        share_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(@class,'fixed bottom-6 right-6')]")
            )
        )
        share_button.click()  # open sidebar
        time.sleep(1)
        share_button.click()  # close sidebar
        print("✅ Floating share button toggled twice")
    except Exception as e:
        print(f"⚠️ Could not toggle share button: {e}")

    # --- 1️⃣7️⃣ Go to Payment Page ---
    driver.get("http://localhost:3000/payment")
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
    time.sleep(4)

    # --- 1️⃣8️⃣ Highlight the created invoice number on Payment page ---
    try:
        invoice_row_payment = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.XPATH, f"//tr[td[contains(text(), '{invoice_id}')]]")
            )
        )
        driver.execute_script(
            """
            arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});
            arguments[0].style.backgroundColor = 'yellow';
            arguments[0].style.fontWeight = 'bold';
        """,
            invoice_row_payment,
        )
        print("✅ Highlighted full invoice row on Payment page (auto-scrolled)")
    except:
        print("⚠️ Invoice row not found on Payment page")

    # --- 1️⃣9️⃣ Go back to Invoice Table Page ---
    driver.get("http://localhost:3000/invoice")
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
    time.sleep(4)

    # --- 2️⃣0️⃣ Highlight the full row of the created invoice ---
    try:
        invoice_row_table = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.XPATH, f"//tr[td[contains(text(), '{invoice_id}')]]")
            )
        )
        driver.execute_script(
            """
            arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});
            arguments[0].style.backgroundColor = '#FFD700';
            arguments[0].style.fontWeight = 'bold';
        """,
            invoice_row_table,
        )
        print("✅ Highlighted full invoice row on Invoice table page (auto-scrolled)")
    except:
        print("⚠️ Invoice row not found on Invoice table page")

    # --- Keep browser open for user to view ---
    input("Press Enter to close the browser...")
    driver.quit()

except Exception as e:
    print(f"⚠️ Error: {e}")
    driver.quit()

✅ Logged in successfully!
✅ Invoice generated! Current URL: http://localhost:3000/invoice/93
✅ Floating share button toggled twice
⚠️ Invoice row not found on Payment page
⚠️ Invoice row not found on Invoice table page
